# ST-GAE: Real-Time Spatiotemporal Anomaly Detection for Toronto Mobility

This notebook implements a **weakly supervised Spatiotemporal Graph Autoencoder (ST-GAE)** for continuous congestion / collision risk scoring on Toronto Bluetooth monitored corridors.

## Approach

Direct collision prediction is unreliable: crashes are rare, sparse, and poorly balanced as labels. Instead we:

1. Learn the **physics of healthy traffic** under ordinary weather and calendar conditions.
2. Score every corridor × 5-minute step by **reconstruction error** $S_{i,t} = \|X_{i,t} - \hat{X}_{i,t}\|^2$.
3. Treat documented collisions as an **evaluation resource** (thresholds, lead-time, PR-AUC), not as scarce training targets.


## Steps

| Step | Section | Purpose |
|---|---|---|
| 1 | Setup, load, clean, normative mask | Reproducible baseline data for training |
| 2 | Feature tensor + corridor graph | Build $X \in \mathbb{R}^{T \times N \times D}$ and adjacency $A$ |
| 3 | ST-GAE train / persist | GCN + GRU autoencoder on normal windows only |
| 4 | Realtime-style scoring | Streaming windows, route thresholds, collision validation |

**Data prerequisite:** upload `route_time_panel_v1.parquet` to Google Drive at `MyDrive/Dissertation/` (built locally with `python -m src.processing --step panel_v1`).

## 0. Colab dependency install

Install the Data Processing + ML stack used by later cells (`pandas`, `numpy`, `sklearn`, `pyarrow`, `matplotlib` `torch`, `torch_geometric`, plus parquet / sklearn helpers).


In [ ]:
# 1. Install standard data science packages and core PyTorch silently
!pip install pandas numpy scikit-learn pyarrow matplotlib torch -q

# 2. Install PyTorch Geometric
!pip install torch-geometric -q

import torch
import torch_geometric

print(
    "Dependencies installed |",
    f"torch={torch.__version__} |",
    f"torch_geometric={torch_geometric.__version__} |",
    f"cuda={torch.cuda.is_available()}",
)

## 1. Environment and configuration

Set up the shared runtime for the rest of the notebook:

- Imports (`numpy`, `pandas`, `pathlib`, etc.) used by cleaning and later modelling cells.
- A single `STGAEConfig` dataclass that holds **all** parameters in one place: Drive paths, train/eval years, rolling window length, model size, learning rate, and normative-mask thresholds.
- Fixed random seeds so tensor construction / training runs are more reproducible across Colab sessions.

**Google Drive layout used**

| Path | Role |
|---|---|
| `/content/drive/MyDrive/Dissertation/` | Project root on Drive |
| `.../route_time_panel_v1.parquet` | Joined analysis panel (input) |
| `.../artifacts/st_gae/` | Saved cleaning QA, scaler, model checkpoints (output) |


**Key config fields (defaults)**

| Field | Default | Meaning |
|---|---|---|
| `train_years` | `(2016,)` | Years used to learn healthy flow |
| `eval_years` | `(2017,)` | Held-out year for anomaly scoring / validation |
| `window_size` | `12` | 12 × 5 min = **60 min** temporal context for the GRU |
| `delay_percentile` | `0.85` | Per-route delay cutoff for the normative mask |
| `precip_threshold_mm` | `2.5` | Heavy-precip cutoff excluded from training baseline |
| `feature_cols` / `scale_cols` | travel, delay, weather, cyclical time, calendar flags | Model inputs; only continuous cols are scaled with **train-only** stats later |


In [ ]:
from __future__ import annotations

import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd

# Google Drive project root as seen from a Colab runtime (/content/drive/...).
DRIVE_ROOT = Path("/content/drive/MyDrive/Dissertation")


@dataclass(frozen=True)
class STGAEConfig:
    # --- Paths ---
    panel_path: Path = DRIVE_ROOT / "route_time_panel_v1.parquet"
    artifact_dir: Path = DRIVE_ROOT / "artifacts" / "st_gae"

    # --- Temporal train / eval protocol ---
    train_years: tuple[int, ...] = (2016,)  # Calendar years used to learn healthy (normative) traffic dynamics.
    eval_years: tuple[int, ...] = (2017,)   # Held-out year(s) for anomaly scoring and collision-based validation.

    # --- Model / training hyperparameters ---
    window_size: int = 12   # Rolling history length in 5-min steps (12 → 60 minutes of context for the GRU).
    batch_size: int = 32    # Number of window samples per optimizer step.
    hidden_dim: int = 64    # Width of GCN / GRU latent state.
    lr: float = 1e-3        # Adam learning rate for reconstruction training.
    epochs: int = 30        # Full passes over the training window dataset.
    seed: int = 42          # Seed for numpy / python RNGs (and torch later) for reproducibility.

    # --- Normative / cleaning thresholds (healthy-flow baseline) ---
    # Delay quantile computed *inside* each context stratum (see delay_context_keys) 
    # so expected rush-hour delay stays "normal" and can be learned by the ST-GAE.
    delay_percentile: float = 0.85

    # Columns that define a delay-context stratum. Keep order stable for QA/debug.
    # - route_id: corridor-specific delay scale
    # - hour: time-of-day (0–23); captures AM/PM peaks
    # - dow: day-of-week (Mon=0 … Sun=6); captures Friday vs Sunday patterns
    delay_context_keys: tuple[str, ...] = ("route_id", "hour", "dow")

    # If a stratum has fewer samples than this, fall back to coarser keys
    # (drop dow → route+hour, then route only) so sparse cells stay stable.
    delay_context_min_count: int = 30
    
    # Precipitation (mm/hour) above this is treated as non-normal weather for training.
    # Kept separate from the delay quantile so rain is an exclusion flag, not mixed into
    # the definition of "typical Rush hour delay".
    precip_threshold_mm: float = 2.5

    # Minimum Bluetooth sample_count required to treat a bin as observed.
    min_sample_count: int = 0

    # --- Feature schema ---
    # Continuous columns scaled with train-only mean/std (no eval leakage).
    scale_cols: tuple[str, ...] = (
        "travel_time_s",  # corridor travel time (seconds)
        "delay_s",  # travel_time_s - free_flow_s
        "wx_temp_c",  # hourly temperature (°C)
        "wx_precip_mm",  # hourly precipitation (mm)
    )

    # Full model input vector at each corridor × time (after cleaning / encoding).
    feature_cols: tuple[str, ...] = (
        "travel_time_s",  # scaled continuous
        "delay_s",  # scaled continuous
        "wx_temp_c",  # scaled continuous
        "wx_precip_mm",  # scaled continuous
        "hour_sin",  # cyclical hour-of-day
        "hour_cos",  # cyclical hour-of-day
        "dow_sin",  # cyclical day-of-week
        "dow_cos",  # cyclical day-of-week
        "is_public_holiday",  # calendar flag (0/1)
        "is_weekend",  # calendar flag (0/1)
    )


# Instantiate
CFG = STGAEConfig()

# Make cleaning / sampling deterministic across re-runs of this notebook.
random.seed(CFG.seed)
np.random.seed(CFG.seed)

print("DRIVE_ROOT:", DRIVE_ROOT)
print("panel_path:", CFG.panel_path)
print("artifact_dir:", CFG.artifact_dir)
print("config:", json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))

# Prefer GPU when Colab provides one; fall back to CPU otherwise.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Mount Google Drive and load the joined analysis panel

Mount Google Drive in Colab, then read `route_time_panel_v1.parquet` from `MyDrive/Dissertation/`.

In [ ]:
from google.colab import drive

# 1. Mount Google Drive (Colab maps it under /content/drive)
drive.mount("/content/drive")

# 2. Load dataset from the Dissertation folder on Drive
file_path = CFG.panel_path

# 3. Create artifacts directory
CFG.artifact_dir.mkdir(parents=True, exist_ok=True)

raw = pd.read_parquet(file_path)
raw["ts_local"] = pd.to_datetime(raw["ts_local"], utc=False)

print("loaded:", file_path)
print("rows:", f"{len(raw):,}")
print("years:", sorted(raw["year"].unique().tolist()))
print("routes:", raw["route_id"].nunique())
print("columns:", list(raw.columns))
display(raw.head(3))
display(raw.isna().mean().sort_values(ascending=False).head(12).to_frame("null_rate"))

## 3. Data cleaning/preprocessing

Deduplicate, enforce dtypes, rebuild calendar encodings, create an **observation mask** so missing weather / travel times are not silently treated as zeros after scaling.

In [ ]:
BOOL_COLS = [
    "is_public_holiday",
    "is_civic_holiday",
    "is_holiday",
    "is_school_break",
    "is_mega_event",
    "is_parade",
    "is_shopping_peak",
    "is_weekend",
]


def clean_panel(df: pd.DataFrame, *, min_sample_count: int) -> pd.DataFrame:
    """Return cleaned_df: modelling-ready panel with observation flags and cyclical time features."""
    cleaned_df = df.copy()

    # One row per corridor × timestamp (panel joins can leave rare dupes).
    before = len(cleaned_df)
    cleaned_df = cleaned_df.drop_duplicates(subset=["ts_local", "route_id"], keep="last")
    n_dupes = before - len(cleaned_df)

    cleaned_df = cleaned_df.sort_values(["ts_local", "route_id"]).reset_index(drop=True)

    # Re-derive calendar fields from local timestamps (DST-safe source of truth).
    cleaned_df["hour"] = cleaned_df["ts_local"].dt.hour.astype("int16")
    cleaned_df["dow"] = cleaned_df["ts_local"].dt.dayofweek.astype("int16")
    cleaned_df["is_weekend"] = cleaned_df["dow"].isin([5, 6])

    cleaned_df["hour_sin"] = np.sin(2 * np.pi * cleaned_df["hour"] / 24.0)
    cleaned_df["hour_cos"] = np.cos(2 * np.pi * cleaned_df["hour"] / 24.0)
    cleaned_df["dow_sin"] = np.sin(2 * np.pi * cleaned_df["dow"] / 7.0)
    cleaned_df["dow_cos"] = np.cos(2 * np.pi * cleaned_df["dow"] / 7.0)

    for col in BOOL_COLS:
        if col in cleaned_df.columns:
            cleaned_df[col] = cleaned_df[col].fillna(False).astype(bool)

    # Observation quality: usable Bluetooth measurement?
    cleaned_df["obs_travel"] = (
        cleaned_df["travel_time_s"].notna()
        & cleaned_df["delay_s"].notna()
        & (cleaned_df["sample_count"].fillna(0) >= min_sample_count)
    )
    cleaned_df["obs_weather"] = cleaned_df["wx_temp_c"].notna()  # precip may be sparse; temp is denser

    # Keep precip NaNs as missing rather than inventing dry weather.
    cleaned_df["wx_precip_mm"] = pd.to_numeric(cleaned_df["wx_precip_mm"], errors="coerce")

    cleaned_df.attrs["n_dupes_dropped"] = int(n_dupes)
    return cleaned_df


panel = clean_panel(raw, min_sample_count=CFG.min_sample_count)
print(
    f"cleaned rows={len(panel):,} | dupes_dropped={panel.attrs['n_dupes_dropped']:,} | "
    f"obs_travel={panel['obs_travel'].mean():.1%} | obs_weather={panel['obs_weather'].mean():.1%}"
)


## 4. Normative training mask (healthy-flow baseline)

Flag corridor-times that represent *ordinary* operating conditions for ST-GAE training.

The outoencoder learns healthy dynamics excluding:
- route-matched collisions (`n_collisions > 0`)
- civic / mega disruptions and road-closing events
- heavy precipitation
- **context-extreme** delay (above the delay percentile *within* the same route × hour × day-of-week)
- unobserved / low-sample bins

**Context-aware delay** A flat per-route percentile would treat Friday 17:00 like Sunday 03:00. Instead we compare each observation to the typical delay distribution for that corridor at that time-of-week. Expected Friday-evening congestion can therefore remain in the training baseline; only unusually bad delays *for that context* are excluded. Sparse strata fall back to coarser keys (`route+hour`, then `route`).

Weather is handled separately via `precip_threshold_mm` (exclusion flag), not mixed into the delay quantile.

Held-out years and inference keep *both* normal and abnormal intervals so reconstruction error can surface risk.

In [ ]:
def context_delay_threshold(
    df: pd.DataFrame,
    *,
    delay_percentile: float,
    context_keys: tuple[str, ...],
    min_count: int,
) -> pd.Series:
    """Per-row delay cutoff using stratified quantiles with coarse fallbacks.

    Primary keys default to (route_id, hour, dow) so Friday evenings are judged
    against other Friday evenings on the same corridor—not against overnight lows.
    """
    thr = pd.Series(np.nan, index=df.index, dtype="float64")
    # Try finest stratum first, then drop trailing keys one at a time.
    for width in range(len(context_keys), 0, -1):
        keys = list(context_keys[:width])
        missing = [k for k in keys if k not in df.columns]
        if missing:
            raise KeyError(f"delay context keys missing from panel: {missing}")

        counts = df.groupby(keys, sort=False)["delay_s"].transform("count")
        quantiles = df.groupby(keys, sort=False)["delay_s"].transform(
            lambda s: s.quantile(delay_percentile)
        )
        fillable = thr.isna() & (counts >= min_count) & quantiles.notna()
        thr = thr.where(~fillable, quantiles)

    # Final fallback: any remaining NaN uses the global percentile on observed delays.
    if thr.isna().any():
        global_thr = float(df["delay_s"].quantile(delay_percentile))
        thr = thr.fillna(global_thr)
    return thr


def compute_normal_mask(
    df: pd.DataFrame,
    *,
    delay_percentile: float,
    precip_threshold_mm: float,
    delay_context_keys: tuple[str, ...] = ("route_id", "hour", "dow"),
    delay_context_min_count: int = 30,
) -> pd.Series:
    """Binary mask: 1 = eligible for normative training loss, 0 = excluded."""
    delay_thr = context_delay_threshold(
        df,
        delay_percentile=delay_percentile,
        context_keys=delay_context_keys,
        min_count=delay_context_min_count,
    )

    no_collision = df["n_collisions"].fillna(0).eq(0)
    no_event_pressure = (
        df["n_events_active"].fillna(0).eq(0)
        & df["n_events_road_close"].fillna(0).eq(0)
        & ~df.get("is_mega_event", False).astype(bool)
        & ~df.get("is_parade", False).astype(bool)
        & ~df.get("is_holiday", False).astype(bool)
    )
    # Missing precip: treat as unknown weather → exclude from *training* baseline
    # (do not pretend it was dry).
    precip = df["wx_precip_mm"]
    normal_weather = precip.notna() & (precip < precip_threshold_mm)
    # Delay is "normal" if it is not extreme *for this corridor/time-of-week context*.
    normal_flow = df["delay_s"].notna() & (df["delay_s"] <= delay_thr)
    observed = df["obs_travel"] & df["obs_weather"]

    return (
        observed & no_collision & no_event_pressure & normal_weather & normal_flow
    ).astype("float32")


panel["is_normal"] = compute_normal_mask(
    panel,
    delay_percentile=CFG.delay_percentile,
    precip_threshold_mm=CFG.precip_threshold_mm,
    delay_context_keys=CFG.delay_context_keys,
    delay_context_min_count=CFG.delay_context_min_count,
)

qa_mask = (
    panel.groupby("year")
    .agg(
        n_rows=("route_id", "size"),
        normal_rate=("is_normal", "mean"),
        collision_rate=("n_collisions", lambda s: float((s.fillna(0) > 0).mean())),
        precip_missing=("wx_precip_mm", lambda s: float(s.isna().mean())),
    )
    .reset_index()
)
display(qa_mask)
print(f"Overall normal baseline rate: {panel['is_normal'].mean():.2%}")
print(
    "delay context:",
    CFG.delay_context_keys,
    "| min_count:",
    CFG.delay_context_min_count,
)


In [ ]:
## 5. Train / eval year split + persist cleaning QA

Slice the cleaned panel into train years and eval years, and write a QA JSON next to model artifacts.

In [ ]:
train_df = panel.loc[panel["year"].isin(CFG.train_years)].copy()
eval_df = panel.loc[panel["year"].isin(CFG.eval_years)].copy()

if train_df.empty or eval_df.empty:
    raise ValueError(
        f"Empty train/eval split. train_years={CFG.train_years}, eval_years={CFG.eval_years}"
    )

cleaning_qa = {
    "panel_path": str(CFG.panel_path),
    "n_rows_all": int(len(panel)),
    "n_routes": int(panel["route_id"].nunique()),
    "train_years": list(CFG.train_years),
    "eval_years": list(CFG.eval_years),
    "n_rows_train": int(len(train_df)),
    "n_rows_eval": int(len(eval_df)),
    "normal_rate_all": float(panel["is_normal"].mean()),
    "normal_rate_train": float(train_df["is_normal"].mean()),
    "normal_rate_eval": float(eval_df["is_normal"].mean()),
    "feature_cols": list(CFG.feature_cols),
    "scale_cols": list(CFG.scale_cols),
    "window_size": CFG.window_size,
    "delay_percentile": CFG.delay_percentile,
    "delay_context_keys": list(CFG.delay_context_keys),
    "delay_context_min_count": CFG.delay_context_min_count,
    "precip_threshold_mm": CFG.precip_threshold_mm,
}
qa_path = CFG.artifact_dir / "cleaning_qa.json"
qa_path.write_text(json.dumps(cleaning_qa, indent=2), encoding="utf-8")

print(f"train: {len(train_df):,} rows | normal={train_df['is_normal'].mean():.2%}")
print(f"eval : {len(eval_df):,} rows | normal={eval_df['is_normal'].mean():.2%}")
print("wrote", qa_path)


## 6. Feature tensorization ($T \times N \times D$)
Convert the tidy corridor×time panel into a dense 3D tensor for the ST-GAE:


In [ ]:
from sklearn.preprocessing import StandardScaler

BOOL_FEATURE_COLS = [c for c in CFG.feature_cols if c.startswith("is_")]
CYCLICAL_COLS = [c for c in CFG.feature_cols if c.endswith("_sin") or c.endswith("_cos")]


def panel_to_grid(df: pd.DataFrame, route_ids: list[str], feature_cols: tuple[str, ...]) -> tuple[pd.DataFrame, pd.DatetimeIndex]:
    """Reindex one year-slice onto a complete (ts_local × route_id) grid."""
    frame = df.drop_duplicates(subset=["ts_local", "route_id"], keep="last").copy()
    time_index = pd.DatetimeIndex(sorted(frame["ts_local"].unique()), name="ts_local")
    grid = pd.MultiIndex.from_product([time_index, route_ids], names=["ts_local", "route_id"])
    gridded = (
        frame.set_index(["ts_local", "route_id"])
        .reindex(grid)
        .reset_index()
    )
    # Missing cells after reindex: not observed / not normal for training loss.
    gridded["is_normal"] = gridded["is_normal"].fillna(0.0).astype("float32")
    gridded["obs_travel"] = gridded["obs_travel"].fillna(False).astype(bool)
    gridded["obs_weather"] = gridded["obs_weather"].fillna(False).astype(bool)
    return gridded, time_index


def fit_scaler_on_train_normal(train_grid: pd.DataFrame, scale_cols: tuple[str, ...]) -> StandardScaler:
    """Fit scaler only on healthy, observed train rows (no eval leakage)."""
    fit_mask = (
        train_grid["is_normal"].eq(1.0)
        & train_grid["obs_travel"]
        & train_grid["obs_weather"]
        & train_grid[list(scale_cols)].notna().all(axis=1)
    )
    fit_rows = train_grid.loc[fit_mask, list(scale_cols)]
    if fit_rows.empty:
        raise ValueError("No rows available to fit StandardScaler (check normative mask).")
    scaler = StandardScaler()
    scaler.fit(fit_rows.to_numpy(dtype=np.float64))
    print(f"scaler fit on {len(fit_rows):,} train-normal rows | cols={list(scale_cols)}")
    return scaler


def apply_scaler_and_stack(
    grid: pd.DataFrame,
    *,
    route_ids: list[str],
    feature_cols: tuple[str, ...],
    scale_cols: tuple[str, ...],
    scaler: StandardScaler,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return X (T,N,D), normal_mask (T,N), obs_mask (T,N)."""
    frame = grid.copy()
    # Scale continuous cols; leave cyclical / boolean flags unscaled.
    scaled = frame[list(scale_cols)].to_numpy(dtype=np.float64)
    # sklearn rejects NaNs: temporarily fill for transform, then restore obs mask.
    nan_mask = np.isnan(scaled)
    scaled_filled = np.where(nan_mask, 0.0, scaled)
    scaled_out = scaler.transform(scaled_filled)
    scaled_out = np.where(nan_mask, 0.0, scaled_out)
    frame.loc[:, list(scale_cols)] = scaled_out

    for col in BOOL_FEATURE_COLS:
        if col in frame.columns:
            frame[col] = frame[col].fillna(False).astype(np.float32)
    for col in CYCLICAL_COLS:
        if col in frame.columns:
            frame[col] = frame[col].fillna(0.0).astype(np.float32)

    T = frame["ts_local"].nunique()
    N = len(route_ids)
    D = len(feature_cols)
    # Ensure route order is stable and complete.
    frame["route_id"] = pd.Categorical(frame["route_id"], categories=route_ids, ordered=True)
    frame = frame.sort_values(["ts_local", "route_id"])

    X = frame[list(feature_cols)].to_numpy(dtype=np.float32).reshape(T, N, D)
    normal_mask = frame["is_normal"].to_numpy(dtype=np.float32).reshape(T, N)
    obs_mask = (
        frame["obs_travel"].to_numpy(dtype=bool) & frame["obs_weather"].to_numpy(dtype=bool)
    ).astype(np.float32).reshape(T, N)
    # Training loss should only see normal ∩ observed.
    train_loss_mask = (normal_mask * obs_mask).astype(np.float32)
    return X, train_loss_mask, obs_mask


# Shared corridor inventory from the full cleaned panel (stable node order).
route_ids = sorted(panel["route_id"].astype(str).unique().tolist())
N = len(route_ids)
print(f"corridors N={N}")

train_grid, train_times = panel_to_grid(train_df, route_ids, CFG.feature_cols)
eval_grid, eval_times = panel_to_grid(eval_df, route_ids, CFG.feature_cols)

scaler = fit_scaler_on_train_normal(train_grid, CFG.scale_cols)
X_train, mask_train, obs_train = apply_scaler_and_stack(
    train_grid,
    route_ids=route_ids,
    feature_cols=CFG.feature_cols,
    scale_cols=CFG.scale_cols,
    scaler=scaler,
)
X_eval, mask_eval, obs_eval = apply_scaler_and_stack(
    eval_grid,
    route_ids=route_ids,
    feature_cols=CFG.feature_cols,
    scale_cols=CFG.scale_cols,
    scaler=scaler,
)

print("X_train", X_train.shape, "| mask_train", mask_train.shape, f"| normal∩obs={mask_train.mean():.2%}")
print("X_eval ", X_eval.shape, "| mask_eval ", mask_eval.shape, f"| normal∩obs={mask_eval.mean():.2%}")

# Persist transforms for realtime / Colab restarts.
import pickle

artifact_meta = {
    "route_ids": route_ids,
    "feature_cols": list(CFG.feature_cols),
    "scale_cols": list(CFG.scale_cols),
    "train_years": list(CFG.train_years),
    "eval_years": list(CFG.eval_years),
    "window_size": CFG.window_size,
}
(CFG.artifact_dir / "tensor_meta.json").write_text(
    json.dumps(artifact_meta, indent=2), encoding="utf-8"
)
with (CFG.artifact_dir / "scaler.pkl").open("wb") as handle:
    pickle.dump(scaler, handle)
print("wrote", CFG.artifact_dir / "scaler.pkl")
print("wrote", CFG.artifact_dir / "tensor_meta.json")


## 7. Physical corridor graph (adjacency $A$)

Build a directed graph over **corridors** (`route_id` nodes). An edge $u \rightarrow v$ exists when corridor $u$’s destination detector matches corridor $v$’s origin detector (head–tail connectivity from `SOURCE_TARGET` IDs).

In [ ]:
def split_route_id(route_id: str) -> tuple[str, str]:
    """Parse Bluetooth corridor id 'SOURCE_TARGET' into detector endpoints."""
    parts = str(route_id).split("_")
    if len(parts) != 2 or not parts[0] or not parts[1]:
        raise ValueError(f"Expected SOURCE_TARGET route_id, got {route_id!r}")
    return parts[0], parts[1]


def build_corridor_edge_index(route_ids: list[str]) -> torch.Tensor:
    """Return edge_index shape (2, E) for PyG from head–tail detector matching."""
    endpoints = {rid: split_route_id(rid) for rid in route_ids}
    # Map detector -> corridor indices that *start* at that detector.
    starts_at: dict[str, list[int]] = {}
    for idx, rid in enumerate(route_ids):
        src, _tgt = endpoints[rid]
        starts_at.setdefault(src, []).append(idx)

    sources: list[int] = []
    targets: list[int] = []
    for i, rid in enumerate(route_ids):
        _src, tgt = endpoints[rid]
        for j in starts_at.get(tgt, []):
            # Directed succession: finish corridor i, then start corridor j.
            if i == j:
                continue
            sources.append(i)
            targets.append(j)

    if not sources:
        raise ValueError(
            "No head–tail corridor links found; check route_id naming (SOURCE_TARGET)."
        )

    edge_index = torch.tensor([sources, targets], dtype=torch.long)
    return edge_index


edge_index = build_corridor_edge_index(route_ids).to(device)
print(
    f"edge_index {tuple(edge_index.shape)} | "
    f"E={edge_index.shape[1]} | denser-than-line-ok for branching corridors"
)
# Quick structural QA: degree summary on CPU
deg = torch.bincount(edge_index[0].cpu(), minlength=N).float()
print(
    f"out-degree mean={deg.mean():.2f} | "
    f"median={deg.median():.0f} | max={deg.max():.0f} | isolates={(deg == 0).sum().item()}"
)

torch.save(edge_index.cpu(), CFG.artifact_dir / "edge_index.pt")
print("wrote", CFG.artifact_dir / "edge_index.pt")


## 8. Sliding-window datasets for ST-GAE

Wrap tensors into fixed-length windows of size `CFG.window_size` (default 12 → 60 minutes). Each sample is $(X_{t:t+W}, M_{t:t+W})$.

In [ ]:
from torch.utils.data import Dataset, DataLoader


class STWindowDataset(Dataset):
    """Sliding windows over a (T, N, D) tensor with a (T, N) loss mask."""

    def __init__(self, X: np.ndarray, mask: np.ndarray, window_size: int):
        if X.ndim != 3:
            raise ValueError(f"X must be (T,N,D), got {X.shape}")
        if mask.shape != X.shape[:2]:
            raise ValueError(f"mask shape {mask.shape} != {X.shape[:2]}")
        if len(X) <= window_size:
            raise ValueError(f"Need T > window_size; T={len(X)}, window={window_size}")
        self.X = torch.from_numpy(np.asarray(X, dtype=np.float32))
        self.mask = torch.from_numpy(np.asarray(mask, dtype=np.float32))
        self.window_size = int(window_size)

    def __len__(self) -> int:
        return int(self.X.shape[0]) - self.window_size

    def __getitem__(self, idx: int):
        sl = slice(idx, idx + self.window_size)
        # x: (W, N, D), m: (W, N)
        return self.X[sl], self.mask[sl]


train_dataset = STWindowDataset(X_train, mask_train, CFG.window_size)
eval_dataset = STWindowDataset(X_eval, mask_eval, CFG.window_size)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.batch_size,
    shuffle=True,
    drop_last=True,
)
# Chronological loader for scoring / realtime-style sweeps.
eval_loader = DataLoader(
    eval_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    drop_last=False,
)

print(
    f"train windows={len(train_dataset):,} | "
    f"eval windows={len(eval_dataset):,} | "
    f"batch_size={CFG.batch_size} | window={CFG.window_size}"
)
xb, mb = next(iter(train_loader))
print(f"sample batch x={tuple(xb.shape)} mask={tuple(mb.shape)} (B,W,N,D) / (B,W,N)")


## 9. ST-GAE architecture (GCN + GRU autoencoder)

A spatiotemporal graph autoencoder that reconstructs each window of corridor features.

1. **Spatial encoder (GCN):** at each time step, every corridor aggregates information from head–tail neighbours under `edge_index`.
2. **Temporal encoder (GRU):** for each corridor, the GCN outputs over the window are treated as a sequence so the model learns short-term evolution (default 60 minutes).
3. **Decoder:** a small MLP maps the latent state back to the $D$-dimensional feature vector $\hat{X}$.

In [ ]:
import torch.nn as nn
from torch_geometric.nn import GCNConv


def batch_edge_index(edge_index: torch.Tensor, batch_size: int, num_nodes: int) -> torch.Tensor:
    """Tile a single graph into a disconnected batch of identical copies for PyG.

    edge_index: (2, E) on the correct device.
    Returns: (2, E * batch_size) with node ids offset by 0, N, 2N, ...
    """
    e = edge_index.size(1)
    # offsets: (B, 1) -> broadcast add to (2, E) then flatten batch
    offsets = torch.arange(batch_size, device=edge_index.device).view(-1, 1) * num_nodes
    # (B, 2, E) -> (2, B*E)
    batched = edge_index.unsqueeze(0) + offsets.view(-1, 1, 1)
    return batched.permute(1, 0, 2).reshape(2, -1)


class STEncoder(nn.Module):
    """GCN over corridors at each time, then GRU over the window per corridor."""

    def __init__(self, num_features: int, hidden_dim: int):
        super().__init__()
        self.gcn = GCNConv(num_features, hidden_dim)
        self.gru = nn.GRU(hidden_dim, hidden_dim, batch_first=True)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        # x: (B, W, N, F)
        bsz, window, n_nodes, n_feat = x.size()
        batched_edges = batch_edge_index(edge_index, bsz, n_nodes)

        spatial = []
        for t in range(window):
            xt = x[:, t, :, :].reshape(bsz * n_nodes, n_feat)
            ht = torch.relu(self.gcn(xt, batched_edges)).view(bsz, n_nodes, -1)
            spatial.append(ht)
        spatial = torch.stack(spatial, dim=1)  # (B, W, N, H)

        # GRU over time for each node: (B*N, W, H)
        gru_in = spatial.permute(0, 2, 1, 3).reshape(bsz * n_nodes, window, -1)
        gru_out, _ = self.gru(gru_in)
        encoded = gru_out.view(bsz, n_nodes, window, -1).permute(0, 2, 1, 3)
        return encoded  # (B, W, N, H)


class STGAE(nn.Module):
    """Spatiotemporal graph autoencoder: encode then reconstruct features."""

    def __init__(self, num_features: int, hidden_dim: int):
        super().__init__()
        self.encoder = STEncoder(num_features, hidden_dim)
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_features),
        )

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        z = self.encoder(x, edge_index)
        return self.decoder(z)


num_features = X_train.shape[-1]
model = STGAE(num_features=num_features, hidden_dim=CFG.hidden_dim).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(
    f"STGAE on {device} | F={num_features} H={CFG.hidden_dim} | "
    f"trainable params={n_params:,}"
)
print(model)

## 10. Train with masked reconstruction MSE

Minimize MSE between $X$ and $\hat{X}$, on cells where the training mask is 1 (normal ∩ observed). Disrupted / missing cells do not contribute to the gradient.

In [ ]:
import matplotlib.pyplot as plt

# Reproducible init for a fresh run. A resumed run keeps the loaded weights as is.
torch.manual_seed(CFG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.seed)

optimizer = torch.optim.Adam(model.parameters(), lr=CFG.lr)

latest_ckpt_path = CFG.artifact_dir / "st_gae_checkpoint.pt"
best_ckpt_path = CFG.artifact_dir / "st_gae_best.pt"
# How often to write a numbered snapshot to storage.
ckpt_every = int(getattr(CFG, "ckpt_every", 5))


def masked_mse(
    x_hat: torch.Tensor,
    x: torch.Tensor,
    mask: torch.Tensor,
) -> torch.Tensor | None:
    """Mean squared error over mask==1 cells (averaged across features too).

    x_hat, x: (B, W, N, F)
    mask:     (B, W, N) with 1 = include in the loss (normal ∩ observed)
    Returns None when a batch has no valid cells (skip the optimizer step).
    """
    mask_f = mask.unsqueeze(-1)  # (B, W, N, 1) broadcasts over F
    n_cells = mask.sum()
    if n_cells <= 0:
        return None
    # Divide by n_cells * F so the scale matches per-feature MSE on masked cells.
    return ((x_hat - x) ** 2 * mask_f).sum() / (n_cells * x.size(-1))


@torch.no_grad()
def evaluate_masked_loss(loader: DataLoader) -> float:
    """Held-out masked MSE. No gradients — used only to pick the best checkpoint."""
    model.eval()
    total = 0.0
    n_batches = 0
    for x_batch, mask_batch in loader:
        x_batch = x_batch.to(device)
        mask_batch = mask_batch.to(device)
        x_hat = model(x_batch, edge_index)
        loss = masked_mse(x_hat, x_batch, mask_batch)
        if loss is None:
            continue
        total += float(loss.item())
        n_batches += 1
    return total / n_batches if n_batches else float("nan")


def make_checkpoint(epoch: int) -> dict:
    """Everything needed to resume training or run §11 scoring after a Colab reset."""
    return {
        "epoch": epoch,  # last *finished* epoch (1-based)
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": {k: list(v) for k, v in history.items()},
        "best_eval": best_eval,
        "num_features": num_features,
        "hidden_dim": CFG.hidden_dim,
        "window_size": CFG.window_size,
        "feature_cols": list(CFG.feature_cols),
        "route_ids": route_ids,
        "train_years": list(CFG.train_years),
        "eval_years": list(CFG.eval_years),
        "seed": CFG.seed,
    }


def load_torch_ckpt(path: Path) -> dict:
    """torch.load helper: weights_only= is only in recent PyTorch."""
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def find_resume_path() -> Path | None:
    """Prefer the latest running checkpoint, then best, then the highest numbered snapshot."""
    if latest_ckpt_path.exists():
        return latest_ckpt_path
    if best_ckpt_path.exists():
        return best_ckpt_path
    numbered = sorted(CFG.artifact_dir.glob("st_gae_epoch_*.pt"))
    return numbered[-1] if numbered else None


def checkpoint_matches_run(ckpt: dict) -> bool:
    """Refuse to resume if the saved net was trained on a different graph / feature schema."""
    ckpt_f = int(ckpt.get("num_features", num_features))
    ckpt_h = int(ckpt.get("hidden_dim", CFG.hidden_dim))
    ckpt_feats = list(ckpt.get("feature_cols") or CFG.feature_cols)
    ckpt_routes = list(ckpt.get("route_ids") or route_ids)
    if ckpt_f != num_features or ckpt_h != CFG.hidden_dim:
        print(f"skip resume: architecture mismatch (ckpt F={ckpt_f} H={ckpt_h}, now F={num_features} H={CFG.hidden_dim})")
        return False
    if ckpt_feats != list(CFG.feature_cols):
        print("skip resume: feature_cols differ from this notebook run")
        return False
    if ckpt_routes != list(route_ids):
        print("skip resume: route_ids / node order differ from this notebook run")
        return False
    return True


# Resume from Drive if a compatible checkpoint exists
history = {"train_loss": [], "eval_loss": []}
best_eval = float("inf")
start_epoch = 1  # first epoch we will *run* (1-based)

resume_path = find_resume_path()
if resume_path is not None:
    ckpt = load_torch_ckpt(resume_path)
    if checkpoint_matches_run(ckpt):
        model.load_state_dict(ckpt["model_state_dict"])
        # Optimizer state is optional: older files may not have it.
        if "optimizer_state_dict" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        history = ckpt.get("history") or history
        history.setdefault("train_loss", [])
        history.setdefault("eval_loss", [])
        finished = int(ckpt.get("epoch", len(history["train_loss"])))
        # Restore the best eval seen so far so we do not overwrite st_gae_best.pt with a worse epoch.
        if history["eval_loss"]:
            best_eval = float(min(history["eval_loss"]))
        if ckpt.get("best_eval") is not None:
            best_eval = min(best_eval, float(ckpt["best_eval"]))
        start_epoch = finished + 1
        print(
            f"resumed {resume_path.name} | finished epoch={finished} | "
            f"best_eval={best_eval:.6f} | next epoch={start_epoch}"
        )
    else:
        print(f"found {resume_path.name} but it does not match this run — training from scratch")
else:
    print("no checkpoint on Drive — training from scratch")

# Nothing left to do if we already hit the configured budget.
if start_epoch > CFG.epochs:
    print(f"already at epoch {start_epoch - 1} ≥ CFG.epochs={CFG.epochs} — skip training loop")
else:
    print(
        f"train epochs {start_epoch}→{CFG.epochs} | lr={CFG.lr} | "
        f"batch={CFG.batch_size} | numbered snapshot every {ckpt_every} epochs"
    )
    print(f"checkpoints → {CFG.artifact_dir}")

    for epoch in range(start_epoch, CFG.epochs + 1):
        model.train()
        running = 0.0
        valid = 0
        for x_batch, mask_batch in train_loader:
            x_batch = x_batch.to(device)
            mask_batch = mask_batch.to(device)

            optimizer.zero_grad(set_to_none=True)
            x_hat = model(x_batch, edge_index)
            loss = masked_mse(x_hat, x_batch, mask_batch)
            if loss is None:
                continue
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running += float(loss.item())
            valid += 1

        train_loss = running / valid if valid else float("nan")
        eval_loss = evaluate_masked_loss(eval_loader)
        history["train_loss"].append(train_loss)
        history["eval_loss"].append(eval_loss)

        ckpt = make_checkpoint(epoch)
        # Always keep a rolling "latest" so a mid-run disconnect can resume.
        torch.save(ckpt, latest_ckpt_path)

        # Numbered snapshots: every ckpt_every epochs, and always the last epoch.
        snapshot = (epoch % ckpt_every == 0) or (epoch == CFG.epochs)
        if snapshot:
            epoch_path = CFG.artifact_dir / f"st_gae_epoch_{epoch:02d}.pt"
            torch.save(ckpt, epoch_path)
            snap_msg = f" | snapshot {epoch_path.name}"
        else:
            snap_msg = ""

        improved = eval_loss < best_eval
        if improved:
            best_eval = eval_loss
            torch.save(ckpt, best_ckpt_path)

        print(
            f"Epoch {epoch:02d}/{CFG.epochs} | "
            f"train_masked_mse={train_loss:.6f} | eval_masked_mse={eval_loss:.6f}"
            + (" | new best" if improved else "")
            + snap_msg
        )

# Train vs eval curves (overfitting check), including resumed history ---
if history["train_loss"]:
    epochs_axis = list(range(1, len(history["train_loss"]) + 1))
    train_curve = history["train_loss"]
    eval_curve = history["eval_loss"]
    gap_curve = [e - t for t, e in zip(train_curve, eval_curve)]
    best_epoch = int(min(range(len(eval_curve)), key=lambda i: eval_curve[i]) + 1)

    fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
    ax0.plot(epochs_axis, train_curve, marker="o", label="train (reconstruction)")
    ax0.plot(epochs_axis, eval_curve, marker="o", label="eval (generalization)")
    ax0.axvline(best_epoch, color="gray", ls="--", lw=1, label=f"best eval (epoch {best_epoch})")
    ax0.set_ylabel("masked MSE")
    ax0.set_title("Reconstruction vs generalization over epochs")
    ax0.legend()
    ax0.grid(True, alpha=0.3)

    ax1.plot(epochs_axis, gap_curve, marker="o", color="C3", label="eval − train")
    ax1.axhline(0.0, color="black", lw=0.8)
    ax1.axvline(best_epoch, color="gray", ls="--", lw=1)
    ax1.set_xlabel("epoch")
    ax1.set_ylabel("generalization gap")
    ax1.set_title("Overfitting signal")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("wrote latest →", latest_ckpt_path)
    if best_ckpt_path.exists():
        print("wrote best   →", best_ckpt_path, f"(eval_masked_mse={best_eval:.6f})")
else:
    print("no loss history to plot")


## 11. Baseline Reconstruction Error (Healthy Flow)

To establish the model's baseline predictive accuracy. A low Root Mean Square Error (RMSE) on strictly normal, observed windows confirms that the GCN and GRU layers successfully learned the physical, healthy dynamics of the corridor network. We inverse-transform the scaled MSE to report this error in physical units (seconds).

In [ ]:
import math

@torch.no_grad()
def evaluate_baseline_rmse(loader, model, edge_idx, feature_cols, scale_cols, scaler):
    """Calculates the physical RMSE for travel time and delay on healthy flow only."""
    model.eval()
    
    # Locate feature indices in the full tensor and the scaler
    tt_idx = feature_cols.index("travel_time_s")
    delay_idx = feature_cols.index("delay_s")
    
    tt_scale_idx = scale_cols.index("travel_time_s")
    delay_scale_idx = scale_cols.index("delay_s")
    
    tt_se = []
    delay_se = []
    normal_mask_list = []
    
    for x_batch, mask_batch in loader:
        x_batch = x_batch.to(device)
        mask_batch = mask_batch.to(device) # mask_batch is 1 for normal ∩ observed[cite: 1]
        
        x_hat = model(x_batch, edge_idx)
        
        # Extract the final timestep of the predictive window
        x_last = x_batch[:, -1, :, :]
        x_hat_last = x_hat[:, -1, :, :]
        mask_last = mask_batch[:, -1, :] 
        
        # Calculate squared error in the scaled latent space
        se_tt = (x_hat_last[:, :, tt_idx] - x_last[:, :, tt_idx]) ** 2
        se_delay = (x_hat_last[:, :, delay_idx] - x_last[:, :, delay_idx]) ** 2
        
        tt_se.append(se_tt.cpu().numpy())
        delay_se.append(se_delay.cpu().numpy())
        normal_mask_list.append(mask_last.cpu().numpy())
        
    # Flatten arrays
    tt_se = np.concatenate(tt_se)
    delay_se = np.concatenate(delay_se)
    normal_mask = np.concatenate(normal_mask_list)
    
    # Filter strictly to normal, healthy, observed flow[cite: 1]
    valid_tt_se = tt_se[normal_mask == 1.0]
    valid_delay_se = delay_se[normal_mask == 1.0]
    
    # Mean Squared Error (Scaled)
    mse_tt_scaled = valid_tt_se.mean()
    mse_delay_scaled = valid_delay_se.mean()
    
    # Inverse transform to physical units (seconds)
    # Physical RMSE = Scaled RMSE * Feature Standard Deviation
    tt_std = scaler.scale_[tt_scale_idx]
    delay_std = scaler.scale_[delay_scale_idx]
    
    rmse_tt_sec = math.sqrt(mse_tt_scaled) * tt_std
    rmse_delay_sec = math.sqrt(mse_delay_scaled) * delay_std
    
    return rmse_tt_sec, rmse_delay_sec

# Execute the baseline evaluation
rmse_tt, rmse_delay = evaluate_baseline_rmse(
    eval_loader, 
    model, 
    edge_index, 
    CFG.feature_cols, 
    CFG.scale_cols, 
    scaler
)

print("--- Baseline Reconstruction Error (Healthy Flow) ---")
print(f"Travel Time RMSE: {rmse_tt:.2f} seconds")
print(f"Delay RMSE:       {rmse_delay:.2f} seconds")

# Compare to the overall scaled anomaly score baseline
baseline_eval = valid_eval[valid_eval["is_normal"] == 1]
overall_scaled_rmse = math.sqrt(baseline_eval["anomaly_score"].mean())
print(f"\nOverall Scaled RMSE (All features combined): {overall_scaled_rmse:.4f}")

## 12. Continuous Anomaly Scoring (Reconstruction Error)

Using the model trained on normal flow dynamics, we score the held-out evaluation year. 
The anomaly score $S_{i,t}$ for corridor $i$ at time $t$ is the mean squared error (MSE) of the reconstructed features at the latest step in the rolling window. We map these dense tensor predictions back to the original tidy dataframe for analysis.

In [ ]:
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

@torch.no_grad()
def generate_realtime_scores(loader, model, edge_idx):
    """Passes the eval set through the model and extracts the reconstruction error for the latest timestep in each window."""
    model.eval()
    all_scores = []
    
    for x_batch, _ in loader:
        x_batch = x_batch.to(device)
        x_hat = model(x_batch, edge_idx)
        
        # Extract the last timestep of the window: (B, N, F)
        x_last = x_batch[:, -1, :, :]
        x_hat_last = x_hat[:, -1, :, :]
        
        # Compute MSE across features: (B, N)
        # Unscaled continuous + cyclical + boolean features all contribute to the deviation
        mse_score = F.mse_loss(x_hat_last, x_last, reduction='none').mean(dim=-1)
        all_scores.append(mse_score.cpu().numpy())
        
    return np.concatenate(all_scores, axis=0)

print("Scoring evaluation windows...")
eval_scores = generate_realtime_scores(eval_loader, model, edge_index)

# The sliding window consumes the first (W-1) steps. 
# Window 0 ends at time index W-1.
W = CFG.window_size
# Match lengths handling the exact __len__ of STWindowDataset
target_time_idx = np.arange(W - 1, W - 1 + len(eval_scores))
target_times = eval_times[target_time_idx]

# Map back to a tidy dataframe
score_df = pd.DataFrame(eval_scores, columns=route_ids, index=target_times)
score_df = score_df.stack().reset_index()
score_df.columns = ["ts_local", "route_id", "anomaly_score"]

# Merge anomaly scores with the original ground-truth eval panel
eval_results = pd.merge(eval_df, score_df, on=["ts_local", "route_id"], how="inner")
eval_results["is_collision"] = (eval_results["n_collisions"].fillna(0) > 0).astype(int)

print(f"Scored {len(eval_results):,} corridor-intervals.")
display(eval_results[["ts_local", "route_id", "delay_s", "n_collisions", "anomaly_score"]].head())

## 13. Localized Anomaly Detection & Spatial Signal Decay

Direct collision prediction is unreliable as a primary training objective, so we treat documented crashes as ground truth to evaluate the unsupervised anomaly signal. However, traffic disruptions are spatiotempora, a crash on one corridor physically impacts its neighbors. 

To evaluate anomaly detection with strict spatial localization, we classify the network state into three spatial tiers during evaluation:
1. **Exact Corridor (0-hop):** The specific corridor where the collision was logged.
2. **Adjacent Corridors (1-hop):** Directly connected upstream and downstream corridors during the exact time of the collision.
3. **Normal Flow:** Healthy corridors operating without any logged incident or extreme environmental pressure.

We evaluate:
*   **Spatial Signal Separation:** Does the reconstruction error perfectly decay from the exact crash site, to adjacent corridors, down to normal flow?
*   **Localized PR-AUC:** How well does the anomaly score pinpoint the exact coordinate of the crash, and how much of the model's "false positive" rate is actually valid shockwave detection on adjacent roads?

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, precision_recall_curve

# 1. Map Spatial Neighbors (1-hop Upstream and Downstream)
route_to_idx = {r: i for i, r in enumerate(route_ids)}
idx_to_route = {i: r for i, r in enumerate(route_ids)}

neighbors_map = {}
for r in route_ids:
    idx = route_to_idx[r]
    # Downstream and upstream connections from the edge_index
    downstream = edge_index[1][edge_index[0] == idx].cpu().numpy()
    upstream = edge_index[0][edge_index[1] == idx].cpu().numpy()
    neighbors_map[r] = set([idx_to_route[i] for i in downstream] + [idx_to_route[i] for i in upstream])

# 2. Assign Spatial Tiers
valid_eval = eval_results[eval_results["obs_travel"] == True].dropna(subset=["anomaly_score"]).copy()

# Identify all specific collision coordinates
crashes = valid_eval[valid_eval["is_collision"] == 1][["ts_local", "route_id"]]

# Build a dataframe of adjacent coordinates at the exact time of the crash
adjacent_records = []
for _, row in crashes.iterrows():
    t = row["ts_local"]
    r = row["route_id"]
    for neighbor in neighbors_map[r]:
        adjacent_records.append((t, neighbor))
        
if adjacent_records:
    adj_df = pd.DataFrame(adjacent_records, columns=["ts_local", "route_id"]).drop_duplicates()
    adj_df["is_adjacent"] = 1
    valid_eval = valid_eval.merge(adj_df, on=["ts_local", "route_id"], how="left")
    valid_eval["is_adjacent"] = valid_eval["is_adjacent"].fillna(0).astype(int)
else:
    valid_eval["is_adjacent"] = 0

# Ensure mutually exclusive spatial tiers
valid_eval.loc[valid_eval["is_collision"] == 1, "is_adjacent"] = 0

normal_mask = valid_eval["is_normal"] == 1
adjacent_mask = valid_eval["is_adjacent"] == 1
collision_mask = valid_eval["is_collision"] == 1

# 3. Calculate Localized Precision-Recall
# Task A: Strict Localization (Must predict EXACT corridor coordinate)
binary_strict = valid_eval[normal_mask | collision_mask].copy()
pr_auc_strict = average_precision_score(binary_strict["is_collision"], binary_strict["anomaly_score"])
baseline_strict = binary_strict["is_collision"].mean()

# Task B: Zone Localization (Predicts EXACT corridor OR adjacent shockwave)
binary_zone = valid_eval[normal_mask | adjacent_mask | collision_mask].copy()
binary_zone["is_valid_anomaly"] = binary_zone["is_collision"] | binary_zone["is_adjacent"]
pr_auc_zone = average_precision_score(binary_zone["is_valid_anomaly"], binary_zone["anomaly_score"])
baseline_zone = binary_zone["is_valid_anomaly"].mean()

print(f"--- Localized PR-AUC Performance ---")
print(f"Strict PR-AUC (Exact Corridor):  {pr_auc_strict:.4f} (Baseline: {baseline_strict:.6f})")
print(f"Zone PR-AUC (Exact + Adjacent):  {pr_auc_zone:.4f} (Baseline: {baseline_zone:.6f})")

# 4. Visualize Spatial Signal Separation & PR Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot A: Spatial Signal Decay (KDE)
sns.kdeplot(data=valid_eval[normal_mask], x="anomaly_score", fill=True, color="blue", label="Normal Flow", ax=ax1, log_scale=True, alpha=0.2)
sns.kdeplot(data=valid_eval[adjacent_mask], x="anomaly_score", fill=True, color="orange", label="Adjacent (1-hop Shockwave)", ax=ax1, log_scale=True, alpha=0.3)
sns.kdeplot(data=valid_eval[collision_mask], x="anomaly_score", fill=True, color="red", label="Exact Collision Corridor", ax=ax1, log_scale=True, alpha=0.5)

ax1.set_title("Spatial Decay of ST-GAE Anomaly Signal")
ax1.set_xlabel("Reconstruction Error (Log Scale)")
ax1.set_ylabel("Density")
ax1.legend()

# Plot B: Strict vs Zone PR Curve
prec_strict, rec_strict, _ = precision_recall_curve(binary_strict["is_collision"], binary_strict["anomaly_score"])
prec_zone, rec_zone, _ = precision_recall_curve(binary_zone["is_valid_anomaly"], binary_zone["anomaly_score"])

ax2.plot(rec_strict, prec_strict, color='darkred', lw=2, label=f'Strict Exact Corridor (AUC={pr_auc_strict:.3f})')
ax2.plot(rec_zone, prec_zone, color='darkorange', lw=2, label=f'Impact Zone (Exact+Adj) (AUC={pr_auc_zone:.3f})')

ax2.axhline(baseline_strict, color='darkred', linestyle=':', alpha=0.5)
ax2.axhline(baseline_zone, color='darkorange', linestyle=':', alpha=0.5)

ax2.set_title("Localized Precision-Recall Curves")
ax2.set_xlabel("Recall (Proportion of Anomalies Detected)")
ax2.set_ylabel("Precision (Valid Detections / Total Flagged)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 14. Spatiotemporal Zone Localization

Official collision logs are frequently subject to reporting delays, meaning the physical disruption often occurs minutes before the timestamp recorded in the dataset. Evaluating only the exact 5-minute bin of the logged crash strictly penalizes the model for correctly identifying a collision before it was officially reported.

To evaluate the operational utility of the ST-GAE, we measure Spatiotemporal Zone-Based Top-K Recall: during a 60-minute window leading up to and including the logged collision time, does the exact corridor or any of its 1-hop physical neighbors appear in the Top-K highest anomaly scores across the network?

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# 1. Map Spatial Neighbors (1-hop Upstream and Downstream)
route_to_idx = {r: i for i, r in enumerate(route_ids)}
idx_to_route = {i: r for i, r in enumerate(route_ids)}

neighbors_map = {}
for r in route_ids:
    idx = route_to_idx[r]
    # Downstream and upstream connections from the edge_index
    downstream = edge_index[1][edge_index[0] == idx].cpu().numpy()
    upstream = edge_index[0][edge_index[1] == idx].cpu().numpy()
    neighbors_map[r] = set([idx_to_route[i] for i in downstream] + [idx_to_route[i] for i in upstream])

# 2. Isolate collision events and calculate Spatiotemporal Zone Ranks
collision_events = valid_eval[valid_eval["is_collision"] == 1].copy()

if collision_events.empty:
    print("No observed collisions in the valid evaluation set to rank.")
else:
    zone_ranks = []
    network_sizes = []
    zone_sizes = []
    
    # We evaluate a 60-minute window ending at the logged collision time.
    eval_window_mins = 60 
    
    for _, event in collision_events.iterrows():
        t = event["ts_local"]
        target_route = event["route_id"]
        
        window_start = t - pd.Timedelta(minutes=eval_window_mins)
        
        # Define the spatial impact zone: exact corridor + 1-hop neighbors
        impact_zone = set([target_route]).union(neighbors_map[target_route])
        
        # Get all network scores within this 20-minute temporal window
        window_data = valid_eval[
            (valid_eval["ts_local"] >= window_start) & 
            (valid_eval["ts_local"] <= t)
        ].copy()
        
        if window_data.empty:
            continue
            
        best_rank_in_window = np.inf
        
        # Rank the network independently at each 5-minute step in the window
        for step_t, snapshot in window_data.groupby("ts_local"):
            network_sizes.append(len(snapshot))
            
            # Rank all active corridors (1 = most anomalous)
            ranks = snapshot["anomaly_score"].rank(method="min", ascending=False)
            snapshot_ranked = snapshot.copy()
            snapshot_ranked["rank"] = ranks
            
            # Isolate the ranks of the corridors that fall within the impact zone
            zone_corridors = snapshot_ranked[snapshot_ranked["route_id"].isin(impact_zone)]
            
            if not zone_corridors.empty:
                zone_sizes.append(len(zone_corridors))
                # The best (lowest) rank achieved by ANY corridor in the impact zone at this specific step
                best_step_rank = zone_corridors["rank"].min()
                best_rank_in_window = min(best_rank_in_window, best_step_rank)
        
        if best_rank_in_window != np.inf:
            zone_ranks.append(best_rank_in_window)

    # 3. Calculate Metrics
    zone_ranks = np.array(zone_ranks)
    total_events = len(zone_ranks)
    avg_network = np.mean(network_sizes) if network_sizes else 0
    avg_zone_size = np.mean(zone_sizes) if zone_sizes else 0
    
    top_1 = (zone_ranks <= 1).sum() / total_events
    top_3 = (zone_ranks <= 3).sum() / total_events
    top_5 = (zone_ranks <= 5).sum() / total_events
    top_10 = (zone_ranks <= 10).sum() / total_events
    
    mrr = (1.0 / zone_ranks).mean()
    median_rank = np.median(zone_ranks)
    
    print(f"Spatiotemporal Zone Localization Performance")
    print(f"Total evaluated collisions: {total_events}")
    print(f"Average network size scored per interval: {avg_network:.0f} corridors")
    print(f"Average size of 1-hop impact zone:        {avg_zone_size:.1f} corridors\n")
    print(f"Windowed MRR:                 {mrr:.4f}")
    print(f"Windowed Median Spatial Rank: {median_rank:.1f}")
    print(f"Windowed Top-1 Accuracy:      {top_1:.1%}")
    print(f"Windowed Top-3 Accuracy:      {top_3:.1%}")
    print(f"Windowed Top-5 Accuracy:      {top_5:.1%}")
    print(f"Windowed Top-10 Accuracy:     {top_10:.1%}")
    
    # 4. Plotting
    fig, ax = plt.subplots(figsize=(8, 5))
    metrics = ['Top-1', 'Top-3', 'Top-5', 'Top-10']
    values = [top_1, top_3, top_5, top_10]
    
    sns.barplot(x=metrics, y=values, palette="viridis", ax=ax)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Recall (Proportion of Collisions Localized)")
    ax.set_title(f"Spatiotemporal Zone Localization\n(Exact + 1-Hop Neighbors within a 20-min Window)")
    
    # Add percentage labels on bars
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.1%}", 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha='center', va='bottom', fontsize=12, fontweight='bold', 
                    xytext=(0, 5), textcoords='offset points')
    
    # Random baseline approximation per interval
    random_top_10_chance = min(1.0, 10 * (avg_zone_size / avg_network))
    ax.axhline(random_top_10_chance, color='gray', linestyle='--', label=f'Random Top-10 Baseline per step (~{random_top_10_chance:.1%})')
    
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

## 15. Operational False Alarm Rate (FAR) & Unlabeled Disruptions

A monitoring system is only viable in a traffic control center if it does not overwhelm operators with alert fatigue. Here, we calculate the **False Alarm Rate (FAR)**: how often the system triggers a sustained 20-minute alert (4 consecutive bins > 95th percentile) that does *not* correspond to a documented collision.

Because this is a weakly supervised system, a strict "False Positive" might not be a model failure; it is often a label failure. By analyzing these false alarms against continuous features (extreme delay, heavy precipitation), we can determine how many "false alarms" were actually correct detections of severe, unlogged network disruptions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Calculate Sustained Alerts across the entire evaluation period
alert_percentile = 0.95
steps_required = 4  # 20 minutes
alert_threshold = valid_eval[valid_eval["is_normal"] == 1]["anomaly_score"].quantile(alert_percentile)

# Sort chronologically per route to ensure rolling logic is accurate
eval_sorted = valid_eval.sort_values(["route_id", "ts_local"]).copy()
eval_sorted["is_high"] = eval_sorted["anomaly_score"] > alert_threshold

# Rolling sum of threshold breaches per corridor
eval_sorted["rolling_high"] = eval_sorted.groupby("route_id")["is_high"].transform(
    lambda x: x.rolling(window=steps_required, min_periods=steps_required).sum()
)

# Identify the exact ONSET of a sustained alert (when it transitions from 3 to 4)
# This prevents counting a single 60-minute disruption as multiple distinct alerts.
eval_sorted["alert_onset"] = (eval_sorted["rolling_high"] == steps_required) & (eval_sorted["rolling_high"].shift(1) < steps_required)

all_alert_onsets = eval_sorted[eval_sorted["alert_onset"] == True].copy()

# 2. Cross-reference Alerts with Ground Truth Collisions
crashes = valid_eval[valid_eval["is_collision"] == 1]
false_alarms = []
true_alerts = []

for _, alert in all_alert_onsets.iterrows():
    alert_time = alert["ts_local"]
    alert_route = alert["route_id"]
    
    # Impact zone (exact node + 1-hop neighbors)
    impact_zone = set([alert_route]).union(neighbors_map.get(alert_route, set()))
    
    # Look for any logged collision in this zone within an operational window 
    # (e.g., from 1 hour before to 2 hours after the alert onset)
    window_start = alert_time - pd.Timedelta(hours=1)
    window_end = alert_time + pd.Timedelta(hours=2)
    
    matching_crashes = crashes[
        (crashes["route_id"].isin(impact_zone)) &
        (crashes["ts_local"] >= window_start) &
        (crashes["ts_local"] <= window_end)
    ]
    
    if not matching_crashes.empty:
        true_alerts.append(alert)
    else:
        false_alarms.append(alert)

false_alarms_df = pd.DataFrame(false_alarms)

# 3. Calculate Operational Rates
total_days = (valid_eval["ts_local"].max() - valid_eval["ts_local"].min()).total_seconds() / (24 * 3600)
total_weeks = total_days / 7.0

n_total_alerts = len(all_alert_onsets)
n_true = len(true_alerts)
n_false = len(false_alarms)

far_per_week = n_false / total_weeks if total_weeks > 0 else 0
far_per_day = n_false / total_days if total_days > 0 else 0

print(f"Operational Alert Metrics (Threshold: > {alert_threshold:.4f} for 20 mins)")
print(f"Evaluation Period:         {total_days:.1f} days")
print(f"Total Sustained Alerts:    {n_total_alerts}")
print(f"Mapped to Collisions (TP): {n_true}")
print(f"Unmapped Alerts (FP):      {n_false}")
print(f"\nFalse Alarm Rate:          {far_per_day:.1f} alerts / day across the network")
print(f"False Alarm Rate:          {far_per_week:.1f} alerts / week across the network")

# 4. Unlabeled Disruption Analysis (Why did the model false-alarm?)
if not false_alarms_df.empty:
    # Check if the false alarms coincided with heavy rain (e.g., > 2.5mm/hr)
    # or extreme context delay (delay higher than the local 85th percentile used in training)
    heavy_rain = false_alarms_df[false_alarms_df["wx_precip_mm"] >= CFG.precip_threshold_mm]
    
    # Re-run context delay check for these rows if needed, or simply check if they are "not normal"
    extreme_delay = false_alarms_df[false_alarms_df["is_normal"] == 0]
    
    rain_fp_pct = len(heavy_rain) / n_false
    delay_fp_pct = len(extreme_delay) / n_false
    
    print("\nDiagnostic of 'False Positives' (Unlabeled Disruptions)")
    print(f"False alarms during heavy precipitation: {len(heavy_rain)} ({rain_fp_pct:.1%})")
    print(f"False alarms flagged as non-normal flow: {len(extreme_delay)} ({delay_fp_pct:.1%})")
    
    # Visualization of False Positives vs True Positives
    fig, ax = plt.subplots(figsize=(9, 5))
    
    categories = ['Valid Collision Alert', 'Weather-Induced Anomaly', 'Extreme Congestion (Unlabeled)', 'Unexplained False Alarm']
    
    # We roughly categorize the FPs to show the model isn't just failing
    unexplained = n_false - len(extreme_delay) # Assume extreme delay captures weather too for simplicity
    if unexplained < 0: unexplained = 0
    
    counts = [n_true, len(heavy_rain), len(extreme_delay) - len(heavy_rain), unexplained]
    
    sns.barplot(x=counts, y=categories, palette=["#2ca02c", "#1f77b4", "#ff7f0e", "#d62728"], ax=ax)
    
    ax.set_title(f"Composition of System Alerts over {total_weeks:.1f} Weeks")
    ax.set_xlabel("Number of Sustained Alerts")
    ax.grid(True, alpha=0.3, axis='x')
    
    for i, p in enumerate(ax.patches):
        ax.annotate(f"{counts[i]}", 
                    (p.get_width() + (max(counts)*0.01), p.get_y() + p.get_height() / 2.), 
                    ha='left', va='center', fontsize=11, fontweight='bold')
        
    plt.tight_layout()
    plt.show()